In [ ]:
from PIL import Image
import math
import re
from glob import glob
from datetime import datetime, timedelta

import pandas as pd
import numpy as np

from scipy.optimize import curve_fit
import cv2

from functools import partial

import matplotlib.pyplot as plt
from matplotlib import patches
import plotly.express as px
import plotly.graph_objects as go
import plotly

plotly.offline.init_notebook_mode(connected=True)

In [ ]:
def read_tiff_to_numpy(file_path):
    """
    Read a TIFF image file and convert it to a NumPy array.

    Args:
        file_path (str): The path to the TIFF image file.

    Returns:
        np.ndarray: A NumPy array representing the TIFF image.
    """
    try:
        # Open the TIFF image using PIL
        image = Image.open(file_path)
        if image.mode != 'L':
            image = image.convert('L')
        # Convert the image to a NumPy array
        image_array = np.array(image)
        return image_array
    except Exception as e:
        print(f"Error reading the TIFF file: {e}")
        return None

def find_centroid(img):
    """
    Calculate the centroid coordinates of an object in a binary image.

    Args:
        image (np.ndarray): A binary image represented as a NumPy array.

    Returns:
        tuple: A tuple containing the (y, x) coordinates of the centroid.
    """
    # Calculate centroid
    total = np.sum(img)
    if total == 0:
        raise ValueError("Empty image - all pixel values are zero")
    # Get image dimensions
    height, width = img.shape
    # Create coordinate grids
    x, y = np.indices((width, height))
    # Calculate weighted coordinates
    x_center = int(np.sum(x * img.T) / total)
    y_center = int(np.sum(y * img.T) / total)
    
    return x_center, y_center


def find_lightest_centroid(img):
    """
    Calculate the centroid coordinates of the lightest object in a binary image.

    Args:
        image (np.ndarray): A binary image represented as a NumPy array.

    Returns:
        tuple: A tuple containing the (y, x) coordinates of the centroid.
    """
    # Calculate centroid
    total = np.sum(img)
    if total == 0:
        raise ValueError("Empty image - all pixel values are zero")
    _img = img.copy()
    max_intensity = np.max(_img)
    _img[img!=max_intensity] = 0

    return find_centroid(_img)

def cartesian_to_polar(x, y, c_x, c_y):
    """
    Convert Cartesian coordinates to polar coordinates with a custom origin.

    Args:
        x (np.ndarray): x-coordinates of the points.
        y (np.ndarray): y-coordinates of the points.
        c_x (float): x-coordinate of the custom origin.
        c_y (float): y-coordinate of the custom origin.

    Returns:
        tuple: A tuple containing the radial distances (r) and angles (theta).
    """
    dx = x - c_x
    dy = y - c_y
    r = np.sqrt(dx**2 + dy**2)
    theta = np.arctan2(dy, dx)
    return r, theta

def polar_to_cartesian(r, theta, c_x, c_y):
    """
    Convert polar coordinates to Cartesian coordinates with a custom origin.

    Args:
        r (np.ndarray): Radial distances from the custom origin.
        theta (np.ndarray): Angles in radians.
        c_x (float): x-coordinate of the custom origin.
        c_y (float): y-coordinate of the custom origin.

    Returns:
        tuple: A tuple containing the x-coordinates (x) and y-coordinates (y).
    """
    x = c_x + r * np.cos(theta)
    y = c_y + r * np.sin(theta)
    return x, y

def normalize_data(data):
    """
    Normalize the data to the range [0, 1].
    Args:
        data (np.ndarray): The input data array.
    Returns:
        np.ndarray: The normalized data array.
    """
    min_val = np.min(data)
    max_val = np.max(data)
    if max_val == min_val:
        return data
    normalized_data = (data - min_val) / (max_val - min_val)
    return normalized_data

def gaussian(x, mu, sigma, b):
    """
    Define the Gaussian function.

    Args:
        x (np.ndarray): Input x values.
        A (float): Amplitude of the Gaussian.
        mu (float): Mean of the Gaussian.
        sigma (float): Standard deviation of the Gaussian.
        b(float): noise
    Returns:
        np.ndarray: Output values of the Gaussian function.
    """
    return np.exp(-(x - mu) ** 2 / (2 * sigma ** 2)) + b


def fitting_gaussian(data):
    """
    Fit a Gaussian function to a given data series.
    Args:
        data (np.ndarray): The data series to fit a Gaussian function.
    Returns:
        tuple: A tuple containing the fitted parameters (A, b, mu, sigma) and the fitted curve.
    """
    x_data = np.arange(len(data))
    initial_guess = [np.argmax(data), 10]
    popt, covariance = curve_fit(gaussian, x_data, data, p0=initial_guess)

    return popt, covariance


def calculate_diameter(popt):
    """
    Calculate the diameter at y = 1/e + b for a given data series.

    Args:
        data (np.ndarray): The data series to fit a Gaussian function.

    Returns:
        float: The calculated diameter.
    """
    

    mu, sigma, b = popt
    diameter = 2 * math.sqrt(2 * sigma**2)
    return diameter

def calculate_xy_diameters(image, centroid):
    """
    Calculate the diameters at y = 1/e + b in x and y directions.

    Args:
        image (np.ndarray): The input image array.
        centroid (tuple): The (y, x) coordinates of the centroid.

    Returns:
        tuple: A tuple containing the x-direction diameter and y-direction diameter.
    """
    c_y, c_x = centroid
    # Extract data for x and y directions
    y_data = image[:, c_x]
    x_data = image[c_y, :]

    # Calculate diameters
    x_diameter = calculate_diameter(x_data)
    y_diameter = calculate_diameter(y_data)

    return x_diameter, y_diameter

def extract_radial_data(image, centroid_x, centroid_y, angle):
    """
    Extract data along a radial line from the centroid at a given angle.

    Args:
        image (np.ndarray): The input image array.
        centroid_x (int): The x-coordinate of the centroid.
        centroid_y (int): The y-coordinate of the centroid.
        angle (float): The angle in degrees.

    Returns:
        np.ndarray: The extracted data along the radial line.
    """
    height, width = image.shape
    angle_rad = np.deg2rad(angle)
    max_length = int(max(
        math.sqrt(centroid_x**2 + centroid_y**2),
        math.sqrt((width - centroid_x)**2 + centroid_y**2),
        math.sqrt(centroid_x**2 + (height - centroid_y)**2),
        math.sqrt((width - centroid_x)**2 + (height - centroid_y)**2)
    ))
    distances = np.arange(-max_length, max_length + 1)
    x_coords = np.round(centroid_x + distances * np.cos(angle_rad)).astype(int)
    y_coords = np.round(centroid_y + distances * np.sin(angle_rad)).astype(int)
    valid_mask = (0 <= x_coords) & (x_coords < width) & (0 <= y_coords) & (y_coords < height)
    x_coords = x_coords[valid_mask]
    y_coords = y_coords[valid_mask]
    return image[y_coords, x_coords]


def calculate_diameter_at_angle(image, centroid_x, centroid_y, angle):
    """
    Calculate the diameter at y = 1/e + b at a given angle.

    Args:
        image (np.ndarray): The input image array.
        centroid (tuple): The (y, x) coordinates of the centroid.
        angle (float): The angle in degrees.

    Returns:
        float: The calculated diameter, or None if fitting fails or A <= 0.
    """
    radial_data = extract_radial_data(image, centroid_x, centroid_y, angle)
    return calculate_diameter(radial_data)


def find_spot_border(image):
    """
    处理光斑图片，计算噪声阈值，去除噪声并拟合包含光斑的圆形。

    参数:
    image (numpy.ndarray): 输入的光斑图片，应为单通道灰度图像。

    返回:
    numpy.ndarray: 去除噪声后的图像。
    tuple: 拟合圆形的圆心坐标 (x, y) 和半径。
    """
    # 步骤 2: canny边缘检测
    noise_threshold = np.mean(image) * 0.5
    denoised_image = np.where(image > noise_threshold, image, 0)
    # 步骤 3: 去除噪声
    denoised_image = cv2.fastNlMeansDenoising(denoised_image, None, 10, 7, 21)
    # denoised_image = cv2.Canny(image, 1, 1)

    # 步骤 3: 拟合一个圆形正好包含光斑
    contours, _ = cv2.findContours(denoised_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        # 找到最大的轮廓
        largest_contour = max(contours, key=cv2.contourArea)
        ((x, y), radius) = cv2.minEnclosingCircle(largest_contour)
        center = (int(x), int(y))
        radius = int(radius)
    else:
        center = (0, 0)
        radius = 0

    return denoised_image, (center, radius)

In [ ]:
# display funcs

def display_3d_image(image_array):
    # Generate x, y coordinates
    x = np.arange(image_array.shape[1])
    y = np.arange(image_array.shape[0])
    X, Y = np.meshgrid(x, y)

    # Create a 3D surface plot
    fig = go.Figure(data=[go.Surface(x=X, y=Y, z=image_array)])

    # Update layout
    fig.update_layout(
        title='3D Visualization of Far Spot Image',
        scene=dict(
            xaxis_title='X Coordinate',
            yaxis_title='Y Coordinate',
            zaxis_title='Brightness'
        )
    )

    fig.show()
    
def display_image(image_array, c_x, c_y):
    fig = px.imshow(image_array, color_continuous_scale='gray')
    # Add the centroid as a red spot
    fig.add_trace(go.Scatter(
        x=[c_x],
        y=[c_y],
        mode='markers',
        marker=dict(color='red', size=10),
        name='Centroid'
    ))
    fig.update_layout(title='Image with Centroid Marked')
    fig.show()

In [ ]:
ROOT_DIR = 'data/20250611/20250611002/digitaloptical4Floor'

# Single analysis

In [ ]:
# read tiff images

far_spot_image = read_tiff_to_numpy('data/20250611/20250611002/digitaloptical4Floor/光轴image/20250611 15：53：59(5-80%平顶31)/20250611 15：54：03.08(5-80%平顶31).TIFF')
c_x, c_y = find_lightest_centroid(far_spot_image)
display_image(far_spot_image, c_x, c_y)

## 光轴束腰半径

In [ ]:
# noise = np.max(far_spot_image[:50, :50])
# far_spot_image[far_spot_image < noise] = 0

c_x, c_y = find_lightest_centroid(far_spot_image)
centroid = (c_y, c_x)


fig = px.imshow(far_spot_image, color_continuous_scale='gray')
# Add the centroid as a red spot
fig.add_trace(go.Scatter(
    x=[c_x],
    y=[c_y],
    mode='markers',
    marker=dict(color='red', size=10),
    name='Centroid'
))
fig.update_layout(title='Image with Centroid Marked')
fig.show()

In [ ]:
display_3d_image(far_spot_image)

In [ ]:
normed_image = far_spot_image / np.max(far_spot_image)

y_data = extract_radial_data(normed_image, c_x, c_y, 0)
x_data = np.arange(len(y_data))
initial_guess = [ np.argmax(y_data), 10, 0 ]
# Perform the curve fitting using the least squares method
params, covariance = curve_fit(gaussian, x_data, y_data, p0=initial_guess)
x_fit = np.linspace(np.min(x_data), np.max(x_data), 1000)
y_fit = gaussian(x_fit, *params)

mu, sigma, b = params
diameter = 2 * math.sqrt(2) * sigma

params = dict(zip(['mu', 'sigma', 'b'], [f"{p:.3f}" for p in params]))
print(f"{params}, Diameter: {diameter:.2f}")

# Visualize the original data and the fitted Gaussian curve
plt.figure(figsize=(10, 6))
plt.plot(x_data, y_data, 'bo', label='Original Data')
plt.plot(x_fit, y_fit, 'r-', label='Fitted Gaussian')
plt.axhline(y=1/math.e + b, color='g', linestyle='--', label=f'y = 1/e + {b:.2f}')
plt.xlabel('Y Coordinate')
plt.ylabel('Brightness')
plt.title(f'Gaussian Fit of Image Column Data: {params}')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
calculate_diameter_at_angle(far_spot_image, c_x, c_y, 90)

## 光斑均匀度

In [ ]:
image_path = '/home/zhanghao/Documents/data-mining/data/20250611/20250611002/digitaloptical4Floor/光瞳image/20250611 16：06：02(6-100%平顶24)/20250611 16：06：06.96(6-100%平顶24).TIFF'

near_spot_image = read_tiff_to_numpy(image_path)

c_x, c_y = find_centroid(near_spot_image)

display_image(near_spot_image, c_x, c_y)

In [ ]:
denoised_image, (center, radius) = find_spot_border(near_spot_image)
print(center, radius)
fig, ax = plt.subplots(1)
ax.imshow(near_spot_image, cmap='gray')
circle = patches.Circle(center, radius, linewidth=2, edgecolor='r', facecolor='none')
ax.add_patch(circle)
ax.set_title('Denoised Image with Fitted Circle')
plt.show()


In [ ]:
y_data = extract_radial_data(near_spot_image, center[0], center[1], 0)
non_zero_indices = np.nonzero(y_data)[0]
a = non_zero_indices[0]  # 第一个非 0 元素的下标
b = non_zero_indices[-1]  # 从右侧计算第一个非 0 元素的下标
c = np.mean(y_data[non_zero_indices])  # 非 0 元素的均值


# Sequential analysis

## Brightness

In [ ]:
spot_files = glob(r'data\0611\1F\光轴image\20250611 15：52：11(60%（31束）)\*.TIFF')
len(spot_files)

In [ ]:
def get_highest_bright(file_path):
    img = read_tiff_to_numpy(file_path)
    return np.max(img)

plt.plot([get_highest_bright(file) for file in spot_files])

## centroid shifting

In [ ]:
# load data

data = pd.read_csv('./4f分析/光轴质心20250606 10：23：07(20%平顶31).TXT', encoding='gbk', header=None, sep='\t')
data.columns = ['time', 'x', 'y']

data['time'] = pd.to_datetime(data['time'], format='%Y-%m-%d-%H：%M：%S.%f')
data['distance'] = np.sqrt(data['x']**2 + data['y']**2)

In [ ]:
# Create a figure with 3 subplots
fig, axes = plt.subplots(3, 1, figsize=(10, 15), sharex=True)
# Plot distance from origin
axes[0].plot(data['time'], data['distance'], label='Distance from (0, 0)', color='blue')
axes[0].set_ylabel('Distance')
axes[0].set_title('Distance from Origin Over Time')
axes[0].legend()
axes[0].grid(True)

# Plot x-coordinate changes
axes[1].plot(data['time'], data['x'], label='X-coordinate', color='green')
axes[1].set_ylabel('X-coordinate')
axes[1].set_title('X-coordinate Over Time')
axes[1].legend()
axes[1].grid(True)

# Plot y-coordinate changes
axes[2].plot(data['time'], data['y'], label='Y-coordinate', color='red')
axes[2].set_xlabel('Time')
axes[2].set_ylabel('Y-coordinate')
axes[2].set_title('Y-coordinate Over Time')
axes[2].legend()
axes[2].grid(True)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45)

# Adjust the layout
plt.tight_layout()
plt.show()


In [ ]:
EXP_NAME = '20250611 15：55：47(80%（31束）)'

In [ ]:
far_spot_res = pd.read_pickle(f'data/20250611/1F/光轴image/{EXP_NAME}/{EXP_NAME}.pkl')
far_spot_res['time'] = pd.to_datetime(far_spot_res['time'])
if 'Unnamed: 0' in far_spot_res.columns:
    far_spot_res.drop(columns=['Unnamed: 0'], inplace=True)
    
far_spot_res.info()
far_spot_res.set_index('time', inplace=True)

    
far_spot_res

In [ ]:
target_time_series = far_spot_res.index

def generate_time_sequence(start: datetime, end: datetime):
    """
    生成从起始时间到结束时间的时间序列，相邻时间间隔0.001秒
    
    Args:
        start: 起始时间点（包含）
        end: 结束时间点（包含）
    
    Returns:
        时间序列列表
    """
    if start > end:
        raise ValueError("结束时间必须大于等于起始时间")
    
    time_sequence = []
    current_time = start
    while current_time <= end:
        time_sequence.append(current_time)
        # 每次递增0.01秒（10毫秒）
        current_time += timedelta(milliseconds=10)
    
    return time_sequence

target_time_series = generate_time_sequence(target_time_series[0], target_time_series[-1])
time_index = pd.DataFrame(target_time_series, columns=['time'])

time_index

In [ ]:
far_spot_selected = pd.merge(time_index, far_spot_res, on='time', how='outer')
far_spot_selected[['centroid_x', 'centroid_y']].interpolate(limit_direction='both', method='pchip', inplace=True)
far_spot_selected.interpolate(inplace=True, method='linear')

far_spot_selected.set_index('time', inplace=True)

far_spot_selected = pd.merge(time_index, far_spot_selected, on='time', how='left')
far_spot_selected.set_index('time', inplace=True)
far_spot_selected

In [ ]:
tm_res = pd.read_csv(f'data/20250611/1F/快反镜数据/{EXP_NAME}.TXT', encoding='gbk', sep='\t', header=None)

tm_res.columns = ['time', 'tm_x', 'tm_y']
tm_res['time'] = pd.to_datetime(tm_res.time, format='%Y-%m-%d-%H：%M：%S.%f')
tm_res.info()

tm_res.set_index('time', inplace=True)
tm_res

In [ ]:
tm_selected = pd.merge(time_index, tm_res, on='time', how='outer')
tm_selected[['tm_x','tm_y']].interpolate(method='pchip', limit_direction='both', inplace=True)
tm_selected = pd.merge(time_index, tm_selected, on='time', how='left')
tm_selected.set_index('time')
tm_selected

In [ ]:
merged = pd.merge(far_spot_selected, tm_selected, on='time', how='outer')
merged.to_csv(f'{EXP_NAME}-interpolated.csv', index=False)

merged

In [ ]:
# 

centroid_x = far_spot_selected['centroid_x'].to_numpy()
dt = far_spot_selected['ms_diff'].to_numpy()

n = len(centroid_x)
fft_vals = np.fft.fft(centroid_x - np.mean(centroid_x))  # 去除直流分量
fft_amp = np.abs(fft_vals) / n  # 计算振幅谱（归一化）

# 计算频率轴（单位：Hz）
dt_avg = np.mean(dt) / 1000  # 平均采样间隔（秒）
freq = np.fft.fftfreq(n, d=dt_avg)[:n//2]  # 取正频率部分

# 绘制频谱图
plt.figure()
plt.plot(freq, fft_amp[:n//2])
plt.xlabel('freq (Hz)')
plt.ylabel('Amp')
plt.title(f'centroid_x fft. Max amp @{freq[np.argmax(fft_amp[:n//2])]}')
plt.grid(True)
plt.show()

In [ ]:
merged['centroid_delta_x'] = merged['centroid_x'] - merged['centroid_x'].mean()
merged['centroid_delta_y'] = merged['centroid_y'] - merged['centroid_y'].mean()
merged['centroid_delta'] = (merged['centroid_delta_x'] ** 2 + merged['centroid_delta_y'] ** 2) ** 0.5

merged['tm_delta_x'] = merged['tm_x'] - merged['tm_x'].mean()
merged['tm_delta_y'] = merged['tm_y'] - merged['tm_y'].mean()
merged['tm_delta'] = (merged['tm_delta_x'] ** 2 + merged['tm_delta_y'] ** 2) ** 0.5

merged

In [ ]:
# 傅里叶变换，分析质心x\y，快反镜x\y的抖动频谱



## wavefront change

In [ ]:
wf_df = pd.read_csv('data/20250611/20250611002/digitaloptical4Floor/波前数据/20250611 15：36：59(1-20%平顶31).TXT', delimiter='\t', header=0, encoding='gbk')
wf_df['时间戳'] = pd.to_datetime(wf_df['时间戳'], format='%Y-%m-%d-%H：%M：%S.%f')
wf_df.set_index('时间戳', inplace=True)

wf_df

In [ ]:
resampled_wf_df = wf_df.resample('0.01s').interpolate(method='time').dropna()
resampled_wf_df.resample('0.1s').wf_diff.plot()

In [ ]:
resampled_wf_df.wf_diff.diff().plot()

In [ ]:
from zernike import RZern

def fit_wavefront(zernike_coeffs, grid_size=256):
    """
    基于 Zernike 系数拟合波前

    参数:
    zernike_coeffs (list or np.ndarray): Zernike 系数列表
    grid_size (int): 生成波前网格的大小，默认为 256

    返回:
    np.ndarray: 拟合后的波前
    """
    n = len(zernike_coeffs)
    cart = RZern(n)
    rho = np.linspace(0, 1, grid_size)
    phi = np.linspace(0, 2 * np.pi, grid_size)
    RHO, PHI = np.meshgrid(rho, phi)
    cart.make_cart_grid(RHO, PHI)
    
    wavefront = np.zeros((grid_size, grid_size))
    for j in range(n):
        wavefront += zernike_coeffs[j] * cart.Zk[j]
    
    return wavefront

zernike_coeffs = np.concat(0, wf_df.iloc[0, 3:].values)
wavefront = fit_wavefront(zernike_coeffs)

In [ ]:
wf_df.iloc[0, 3:].values

wf_df = pd.read_csv('data/波前20250606 10：23：07(20%平顶31).TXT', delimiter='\t', header=0)
wf_df

## 功率数据

In [ ]:
data = pd.read_csv('data/功率计数据/20250731 16：58：27(100%平顶18).TXT', encoding='gbk',sep='\t', header=None)
data.columns = ['time', 'power']
data.plot()

In [ ]:
data = pd.read_csv('data/功率计数据/20250815 16：04：41(100%平顶18).TXT', encoding='gbk',sep='\t', header=None)
data.columns = ['time', 'power']
data.plot()

## 大气环境处理

### load data

In [ ]:
import os
# 平均风速 平均温度 平均湿度 平均能见度 计算r0
os.listdir('data/20250611/大气数据/20250611_atmosphere')

In [ ]:
target_time ='''2025/6/6	10:24
2025/6/6	10:36
2025/6/6	10:45
2025/6/6	10:55
2025/6/6	11:26
2025/6/6	11:37
2025/6/6	14:29
2025/6/6	14:49
2025/6/9	11:42
2025/6/9	11:47
2025/6/9	13:26
2025/6/9	13:36
2025/6/9	13:46
2025/6/9	15:09
2025/6/9	15:18
2025/6/10	13:45
2025/6/10	13:52
2025/6/10	14:57
2025/6/10	15:06
2025/6/10	15:56
2025/6/10	16:02
2025/6/11	16:35
2025/6/11	16:47
2025/6/11	16:58
2025/6/12	9:04
2025/6/12	9:13
2025/6/12	9:17
2025/6/12	9:39
2025/6/12	9:42
2025/6/12	11:29
2025/6/12	11:35'''.split('\n')

target_time_df = pd.DataFrame(target_time, columns=['time'])
target_time_df['time'] = pd.to_datetime(target_time_df['time'], format="%Y/%m/%d\t%H:%M")

target_time_df


In [ ]:
def relocate(position, array):
    """
    按预设位置对分层参数进行加权平均（匹配实际大气分层位置）
    
    参数:
        array (list/np.ndarray): 待平均的分层参数数组（长度应等于layers）
        
    返回:
        np.ndarray: 平均后的参数数组（长度等于layers）
    """
    assert len(array) == len(position)
    
    ave = np.zeros_like(array)
    ave[0] = ((array[0] + array[1] + array[2] + array[3] + array[4]) * 20 + array[5] * 100) / 200
    # 后续层线性插值（根据position数组的位置间隔）
    for i in range(15):
        d = (i+2) * 200  # 当前层的位置（间隔200米）
        for j in range(15):
            # 找到d所在的position区间，进行线性插值
            if position[j] <= d <= position[j+1]:
                ave[i+1] = ((d - position[j]) * array[j] + (position[j+1] - d) * array[j+1]) / (position[j+1] - position[j])
            else:
                ave[i+1] = array[-1]  # 超出范围时取最后一个值
    return ave

In [ ]:
wind_df = pd.read_excel(r'data\20250611\大气数据\20250611_atmosphere\Wind\Wind.xlsx')
wind_df['time'] = pd.to_datetime('2025-'+wind_df['m/s'])
wind_df.drop('m/s', axis=1, inplace=True)
wind_df.info()

position = [int(p[:-1]) for p in wind_df.columns if p[:-1].isdigit()]
ave_at_pos = partial(relocate, position)
def ave(array):
    return np.mean(ave_at_pos(array))

wind_df['avg'] = wind_df[[p for p in wind_df.columns if p[:-1].isdigit()]].apply(ave, axis=1)
wind_df['avg'].plot()

In [ ]:
temporature_df = pd.read_excel(r'data\20250611\大气数据\20250611_atmosphere\Temperature\Temperature.xlsx')
temporature_df['time'] = pd.to_datetime('2025-'+temporature_df['°C'])
temporature_df.drop('°C', axis=1, inplace=True)
temporature_df.info()

position = [int(p[:-1]) for p in temporature_df.columns if p[:-1].isdigit()]
ave_at_pos = partial(relocate, position)
def ave(array):
    return np.mean(ave_at_pos(array))

temporature_df['avg'] = temporature_df[[p for p in temporature_df.columns if p[:-1].isdigit()]].apply(ave, axis=1)
temporature_df['avg'].plot()